# House Stage 1: does dropping generic-ballot momentum from Model A improve robustness?

Model A includes `gb_momentum_30d_raw`, the change in the generic-ballot margin over the 30 days before the cutoff. Its fitted coefficient is negative (about −1.3 to −1.7). This notebook asks whether that term earns its place or only reflects an unstable split of the national signal between two collinear national series. It compares:

| model | prediction |
|---|---|
| **A** | current Model A: `partisan_lean + gen_margin_raw + inc_signed` (fixed +2.5) `+ partisan_lean:inc_signed + gb_momentum_30d_raw` |
| **A-NoMom** | identical, without `gb_momentum_30d_raw` |
| **AB-15** | 0.85 A + 0.15 B: the current `house_ensemble_ab_85_15` |
| **AB-NoMom-15** | 0.85 A-NoMom + 0.15 B, with the same lean-inclusive Model B |

**Design:**

- **Fixed specifications:** the established configurations in `house.stage1` are used unchanged. A-NoMom is defined here only. The fixed +2.5 incumbency coefficient, the 0.15 weight, and Model B stay exactly as they are, and nothing is tuned.
- **Estimation:** every model is fit by equal-year WLS (`stage1.fit_predict`), on expanding windows from 2006 through the election before each eval year.
- **Matched samples:** every model fits and scores exactly the same district-years in each window (asserted).
- **Evaluation years:** 2018, 2020, 2022, and 2024. **2024 is development evidence, not a holdout:** `01_reg_final_2024_holdout.ipynb` already scored it.
- **Subsets:** valid targets, ex-ante competitive (|lean + eve-of-election generic ballot| ≤ 10), and ex-post competitive (|actual two-party margin| ≤ 7.5).
- **Residuals:** actual − predicted, D − R points. Positive means Democrats outperformed the model.
- **Averages:** *pooled* weights every race equally; *equal-year mean* averages the four per-year values.
- **Identification:** `gen_margin_raw` and `gb_momentum_30d_raw` are national, with one value per election year. Their coefficients are identified only from differences between years: 6 training years in the 2018 window, up to 9 in the 2024 window. Section 4 examines what that implies.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import house.regression as hr
from house.experiments import evaluation, tracking
from house import stage1
from house.regression.schema import KEY

with warnings.catch_warnings(action="ignore"):  # loading warns about known optional-column gaps
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, 2006-2026
    targets = hr.load_targets()        # data/processed/house/targets.csv, 2006-2024

## Settings and models

In [ ]:
TARGET = stage1.TARGET
EVAL_YEARS = [2018, 2020, 2022, 2024]  # each fit on 2006 through the election before it
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
EX_ANTE_THRESHOLD = 10
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),  # known before the election
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),                     # the observed margin
}
SECTIONS = {"EX_ANTE": "competitive_ex_ante", "EX_POST": "competitive_ex_post", "ALL": "valid"}  # priority order
MOMENTUM = "gb_momentum_30d_raw"
GB = "gen_margin_raw"

# Model A as it was when this comparison was made (2026-11-02, before production dropped momentum): the
# baseline here. Production adopted the result, so stage1.MODEL_A is now Model A without momentum.
MODEL_A_MOM = stage1.Component(
    "A_polling",
    f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed + gb_momentum_30d_raw",
    fixed=dict(stage1.MODEL_A.fixed),
)
AB_15 = stage1.Stage1Config("house_ensemble_ab_85_15_momentum", ((MODEL_A_MOM, 0.85), (stage1.MODEL_B, 0.15)))
A_ALONE = stage1.Stage1Config("house_model_a_momentum", ((MODEL_A_MOM, 1.0),))

# The alternative, defined here only: Model A without momentum, otherwise identical (same fixed incumbency)
MODEL_A_NOMOM = stage1.Component(
    "A_nomom",
    f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed",
    fixed=dict(stage1.MODEL_A.fixed),
)
assert MODEL_A_MOM.formula == MODEL_A_NOMOM.formula + f" + {MOMENTUM}"  # nothing else differs
assert MODEL_A_NOMOM.formula == stage1.MODEL_A.formula  # what production adopted
AB_NOMOM_15 = stage1.Stage1Config("ab_nomom_15", ((MODEL_A_NOMOM, 0.85), (stage1.MODEL_B, 0.15)))
A_NOMOM_ALONE = stage1.Stage1Config("a_nomom", ((MODEL_A_NOMOM, 1.0),))  # only for the leave-one-year-out fits
CONFIGS = {"AB-15": AB_15, "AB-NoMom-15": AB_NOMOM_15}

for label, config in CONFIGS.items():
    print(label, [(c.name, w) for c, w in config.components])
    for component, _ in config.components:
        print(f"    {component.name:15s} {component.formula}   fixed: {dict(component.fixed) or 'none'}")

## Rolling fits

Both ensembles are fit in every window. Their Model B fits must be identical, and every component must use the same training and evaluation rows; both conditions are asserted.

In [ ]:
with warnings.catch_warnings(action="ignore"):
    rolling = {label: stage1.rolling_predictions(config, EVAL_YEARS, predictors=predictors, targets=targets)
               for label, config in CONFIGS.items()}

for year in EVAL_YEARS:
    base, alt = rolling["AB-15"][year], rolling["AB-NoMom-15"][year]
    runs = [*base.components.values(), *alt.components.values()]
    assert len({(r.counts["train_rows_hash"], r.counts["eval_rows_hash"]) for r in runs}) == 1, year
    assert base.predictions[KEY].equals(alt.predictions[KEY]), year
    assert np.array_equal(base.predictions["prediction_B_fundamentals"], alt.predictions["prediction_B_fundamentals"])
    print(year, "trained on", runs[0].config["train_years"][0], "-", runs[0].config["train_years"][-1],
          "| train rows", runs[0].counts["n_train"], "| eval rows", len(base.predictions))

MODELS = ["A", "A-NoMom", "AB-15", "AB-NoMom-15"]
BASE = {"A-NoMom": "A", "AB-15": "A", "AB-NoMom-15": "AB-15"}  # what each model's Δ is measured against
predictions = pd.concat([
    rolling["AB-15"][year].predictions[KEY].assign(**{
        "A": rolling["AB-15"][year].predictions["prediction_A_polling"],
        "A-NoMom": rolling["AB-NoMom-15"][year].predictions["prediction_A_nomom"],
        "AB-15": rolling["AB-15"][year].predictions["prediction"],
        "AB-NoMom-15": rolling["AB-NoMom-15"][year].predictions["prediction"],
    }) for year in EVAL_YEARS], ignore_index=True)

scores = {m: evaluation.evaluate_predictions(predictions[KEY].assign(prediction=predictions[m]), targets, predictors,
                                             TARGET, competitive=COMPETITIVE, race_types=EVAL_RACE_TYPES)
          for m in MODELS}
metrics = {m: s[0] for m, s in scores.items()}
scored = {m: s[2] for m, s in scores.items()}
assert all(scored[m][KEY].equals(scored["A"][KEY]) for m in MODELS)  # identical valid district-years

races = scored["A"][[*KEY, "actual_margin", "competitive_ex_ante", "competitive_ex_post"]].merge(
    predictors[[*KEY, "partisan_lean"]], on=KEY, how="left", validate="one_to_one")
PRED = pd.DataFrame({m: scored[m]["predicted_margin"].to_numpy() for m in MODELS}, index=races.index)
RESID = races["actual_margin"].to_numpy()[:, None] - PRED
MASKS = {"EX_ANTE": races["competitive_ex_ante"], "EX_POST": races["competitive_ex_post"],
         "ALL": pd.Series(True, index=races.index)}
scores["A"][1]  # coverage: districts per eval year by exclusion reason, and the subset sizes

## 1. Performance

MAE and winner accuracy for each subset, per eval year, pooled, and as the equal-year mean. Δ compares each model with its counterpart: **A-NoMom vs A**, **AB-NoMom-15 vs AB-15**, and AB-15 vs A for reference. ✓ marks an improvement and ✗ a regression.

In [ ]:
METRICS = ["n", "mae", "rmse", "accuracy_pct", "missed_races"]
DELTAS = ["mae", "accuracy_pct", "missed_races"]


def performance(period):
    # model x (section, metric); period is an eval year, "pooled", or "equal-year mean"
    def values(m, subset):
        if period == "pooled":
            return metrics[m].loc[(subset, "all")]
        if period == "equal-year mean":
            return sum(metrics[m].loc[(subset, y)] for y in EVAL_YEARS) / len(EVAL_YEARS)
        return metrics[m].loc[(subset, period)]

    rows = {}
    for m in MODELS:
        row = {}
        for section, subset in SECTIONS.items():
            v = values(m, subset)
            row |= {(section, k): v[k] for k in METRICS}
            if m in BASE:
                base = values(BASE[m], subset)
                row |= {(section, f"Δ{k}"): v[k] - base[k] for k in DELTAS}
        rows[f"{m} (Δ vs {BASE[m]})" if m in BASE else m] = row
    return pd.DataFrame.from_dict(rows, orient="index")


for period in ["pooled", "equal-year mean"]:
    display(Markdown(f"### {period.capitalize()}, 2018–2024"))
    display(tracking.style_best_of(performance(period)))

In [ ]:
by_year = pd.concat({year: performance(year) for year in EVAL_YEARS}, names=["eval_year", "model"]).swaplevel()
by_year = by_year.reindex([(m, y) for m in performance(EVAL_YEARS[0]).index for y in EVAL_YEARS])
display(Markdown("### Per eval year"))
tracking.style_best_of(by_year)

### Paired comparison on the same races

The mean change in absolute error from dropping momentum (negative favors NoMom), its naive standard error (races treated as independent, which is optimistic because errors share a national component within each year), the share of races where NoMom is closer, and the number of eval years (of 4) whose MAE favors NoMom.

In [ ]:
paired = {}
for nomom, current in [("A-NoMom", "A"), ("AB-NoMom-15", "AB-15")]:
    diff = RESID[nomom].abs() - RESID[current].abs()
    for section, mask in MASKS.items():
        d = diff[mask]
        per_year = d.groupby(races.loc[mask, "year"]).mean()
        paired[(f"{nomom} vs {current}", section)] = {
            "n": len(d), "mean Δ|residual|": d.mean(), "naive se": d.std() / np.sqrt(len(d)),
            "% races closer": 100 * (d < 0).mean(), "years favoring NoMom": int((per_year < 0).sum()),
            **{f"Δ {y}": per_year[y] for y in EVAL_YEARS}}
pd.DataFrame.from_dict(paired, orient="index").rename_axis(["comparison", "subset"]).style.format(precision=3).format(
    {"n": "{:.0f}", "years favoring NoMom": "{:.0f}", "% races closer": "{:.1f}"})

## 2. Winner calls fixed and broken by dropping momentum

**Fixed** means NoMom is right and the current specification was wrong; **broken** means the reverse. Net = fixed − broken. A race counts in every subset it belongs to. The list below the table shows every competitive race whose call changes.

In [ ]:
def correct(m):
    return pd.Series(np.sign(PRED[m]) == np.sign(races["actual_margin"]), index=races.index)


flip_rows = {}
for nomom, current in [("A-NoMom", "A"), ("AB-NoMom-15", "AB-15")]:
    fixed, broken = correct(nomom) & ~correct(current), ~correct(nomom) & correct(current)
    for year in [*EVAL_YEARS, "pooled"]:
        row = {}
        for section, mask in MASKS.items():
            in_year = mask if year == "pooled" else mask & (races["year"] == year)
            f, b = int((fixed & in_year).sum()), int((broken & in_year).sum())
            row |= {(section, "fixed"): f, (section, "broken"): b, (section, "net"): f - b}
        flip_rows[(f"{nomom} vs {current}", year)] = row
flips = pd.DataFrame.from_dict(flip_rows, orient="index").rename_axis(["comparison", "eval_year"])
display(flips.style.map(lambda v: f"color: {tracking.GOOD}" if v > 0 else f"color: {tracking.BAD}" if v < 0 else "",
                        subset=[c for c in flips.columns if c[1] == "net"]))

competitive = races["competitive_ex_ante"] | races["competitive_ex_post"]
changed = []
for nomom, current in [("A-NoMom", "A"), ("AB-NoMom-15", "AB-15")]:
    for i in races.index[competitive & (np.sign(PRED[nomom]) != np.sign(PRED[current]))]:
        r = races.loc[i]
        changed.append({"comparison": f"{nomom} vs {current}", "year": r["year"],
                        "district": evaluation.district_label(r["state"], r["district"]),
                        "ex_ante": r["competitive_ex_ante"], "ex_post": r["competitive_ex_post"],
                        "actual": r["actual_margin"], "current": PRED.loc[i, current], "NoMom": PRED.loc[i, nomom],
                        "verdict": "fixed" if correct(nomom)[i] else "broken"})
changed = pd.DataFrame(changed, columns=["comparison", "year", "district", "ex_ante", "ex_post", "actual", "current",
                                         "NoMom", "verdict"])
changed.style.format({"actual": "{:+.1f}", "current": "{:+.1f}", "NoMom": "{:+.1f}"}).map(
    lambda v: f"color: {tracking.GOOD}" if v == "fixed" else f"color: {tracking.BAD}" if v == "broken" else "",
    subset=["verdict"])

## 3. Coefficient stability across training windows

Each model's coefficients in each expanding window, their spread, and the WLS standard errors. `inc_signed` is the fixed +2.5. The row-level standard errors treat every race as independent. For the national terms, `gen_margin_raw` and `gb_momentum_30d_raw`, that overstates precision, since their information comes from only 6–9 election years (section 4).

In [ ]:
COMPONENTS = {"A": ("AB-15", "A_polling"), "A-NoMom": ("AB-NoMom-15", "A_nomom")}


def coefficient_table(model, column):
    label, name = COMPONENTS[model]
    return pd.DataFrame({year: rolling[label][year].components[name].coefficients[column] for year in EVAL_YEARS})


coefs = pd.concat({m: coefficient_table(m, "coef") for m in COMPONENTS}, names=["model", "term"])
spread = pd.DataFrame({"mean": coefs.mean(axis=1), "sd": coefs.std(axis=1), "min": coefs.min(axis=1),
                       "max": coefs.max(axis=1)})
spread["sd/|mean|"] = spread["sd"] / spread["mean"].abs()
display(Markdown("### Coefficients by eval window"))
display(coefs.join(spread).style.format(precision=3))
display(Markdown("### Standard errors by eval window (blank: fixed coefficient)"))
pd.concat({m: coefficient_table(m, "std_err") for m in COMPONENTS}, names=["model", "term"]).round(3)

## 4. Is the negative momentum coefficient unstable collinearity?

Both generic-ballot terms take a single value per election year, so the national part of Model A is a regression on a handful of points. The four checks below examine how momentum's coefficient is identified.

1. **The year-level data:** each election's generic ballot, momentum, and national outcome.
2. **Collinearity per training window:** the correlation of `gen_margin_raw` and momentum across the window's training years, the variance inflation factor (VIF = 1 / (1 − r²)), and the year-level residual degrees of freedom left after the intercept and both national slopes.
3. **Leave-one-year-out:** refit both models on each window with one training year dropped. If one or two elections drive momentum's coefficient, it swings, and so does the generic-ballot coefficient it trades off against.
4. **The year-level added-variable plot:** in the 2024 window (2006–2022), each training year's mean in-sample residual from A-NoMom, against momentum. Momentum's coefficient in A is essentially the slope of this cloud, net of the generic ballot.

In [ ]:
year_level = predictors.groupby("year")[[GB, MOMENTUM, "gb_margin_2p_norm"]].first().loc[2006:2024]
national = targets[targets["year"] <= 2024].merge(predictors[[*KEY, "partisan_lean"]], on=KEY)
national = national[~national["imputation_method"].isin(hr.schema.MODELED_IMPUTATION_METHODS)]
year_level["mean(actual − lean)"] = (national[TARGET] - national["partisan_lean"]).groupby(national["year"]).mean()
display(Markdown("### The national series, one value per election"))
display(year_level.style.format(precision=2))

collinearity = {}
for year in EVAL_YEARS:
    train = year_level.loc[stage1.expanding_window(stage1.HOUSE_MODEL_A, year)]
    r = train[GB].corr(train[MOMENTUM])
    collinearity[year] = {"training years": len(train), "corr(gb, momentum)": r, "VIF": 1 / (1 - r ** 2),
                          "year-level resid df": len(train) - 3}
display(Markdown("### Collinearity of the two national series, per training window"))
pd.DataFrame.from_dict(collinearity, orient="index").rename_axis("eval window").style.format(precision=2).format(
    {"training years": "{:.0f}", "year-level resid df": "{:.0f}"})

In [ ]:
loyo = []
with warnings.catch_warnings(action="ignore"):
    for year in EVAL_YEARS:
        window = stage1.expanding_window(stage1.HOUSE_MODEL_A, year)
        for dropped in [None, *window]:
            years = [y for y in window if y != dropped]
            row = {"eval window": year, "dropped": dropped if dropped else "none"}
            for label, config, name in [("A", A_ALONE, "A_polling"), ("A-NoMom", A_NOMOM_ALONE, "A_nomom")]:
                coef = stage1.fit_predict(config, years, year, predictors=predictors,
                                          targets=targets).components[name].coefficients["coef"]
                row[f"{label}: {GB}"] = coef[GB]
                if label == "A":
                    row[f"A: {MOMENTUM}"] = coef[MOMENTUM]
            loyo.append(row)
loyo = pd.DataFrame(loyo)
columns = [f"A: {MOMENTUM}", f"A: {GB}", f"A-NoMom: {GB}"]
loyo_range = loyo[loyo["dropped"] != "none"].groupby("eval window")[columns].agg(["min", "max"])
for column in columns:
    loyo_range[(column, "full")] = loyo[loyo["dropped"] == "none"].set_index("eval window")[column]
    loyo_range[(column, "range")] = loyo_range[(column, "max")] - loyo_range[(column, "min")]
loyo_range = loyo_range[[(c, s) for c in columns for s in ["full", "min", "max", "range"]]]
display(Markdown("### Leave-one-year-out: the coefficient on the full window, and its min / max / range over the "
                 "one-year-out refits"))
display(loyo_range.style.format(precision=2))
display(Markdown("### Leave-one-year-out detail: which dropped year moves momentum's coefficient most"))
pivot = loyo.pivot(index="dropped", columns="eval window", values=f"A: {MOMENTUM}")
pivot.style.format(precision=2, na_rep="")

In [ ]:
COLORS = {"A": "#2a78d6", "A-NoMom": "#eb6834", "AB-15": "#1baf7a", "AB-NoMom-15": "#eda100"}
STYLES = {"A": "-", "A-NoMom": "--", "AB-15": "-", "AB-NoMom-15": "--"}


def tidy(ax, title, xlabel, ylabel):
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors="#52514e")


fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
# left: coefficients per window, with the leave-one-year-out range as a bar
for offset, (column, color, label) in enumerate([(f"A: {MOMENTUM}", COLORS["A"], "A: momentum"),
                                                 (f"A: {GB}", "#6250d6", "A: generic ballot"),
                                                 (f"A-NoMom: {GB}", COLORS["A-NoMom"], "A-NoMom: generic ballot")]):
    x = np.array(EVAL_YEARS) + (offset - 1) * 0.35
    full, low, high = (loyo_range[(column, s)].to_numpy() for s in ["full", "min", "max"])
    axes[0].vlines(x, low, high, color=color, lw=2, alpha=0.6)
    axes[0].plot(x, full, "o", color=color, ms=8, label=label)
axes[0].axhline(0, color="#8d8c86", lw=1)
axes[0].set_xticks(EVAL_YEARS)
tidy(axes[0], "Coefficient per window (dot) and its leave-one-year-out range (bar)", "eval window", "coefficient")
axes[0].legend(frameon=False, fontsize=9)

# right: the 2024 window's year-level added-variable view
run = rolling["AB-NoMom-15"][2024].components["A_nomom"]
train_rows = run.predictions[run.predictions["role"] == "train"]
year_resid = train_rows.groupby("year")["residual"].mean()
mom = year_level.loc[year_resid.index, MOMENTUM]
gb = year_level.loc[year_resid.index, GB]
mom_net = mom - np.polyval(np.polyfit(gb, mom, 1), gb)  # momentum net of the generic ballot, across years
axes[1].axhline(0, color="#8d8c86", lw=1)
axes[1].plot(mom_net, year_resid, "o", color=COLORS["A"], ms=8)
for y in year_resid.index:
    axes[1].annotate(str(y), (mom_net[y], year_resid[y]), textcoords="offset points", xytext=(6, 4), fontsize=9,
                     color="#52514e")
slope = np.polyfit(mom_net, year_resid, 1)[0]
grid = np.linspace(mom_net.min(), mom_net.max(), 10)
axes[1].plot(grid, slope * grid + year_resid.mean(), color=COLORS["A"], lw=2, alpha=0.5)
tidy(axes[1], f"2024 window: A-NoMom's mean residual by training year (slope {slope:+.2f})",
     "momentum, net of the generic ballot (pts)", "mean in-sample residual (pts)")
plt.tight_layout()

## 5. National directional bias

Each year's mean residual: the error every district shares that year. Momentum is national, so dropping it moves every prediction in an eval year by almost the same amount. The second table shows that shift directly: the mean and the within-year standard deviation of A-NoMom − A, next to momentum's contribution in A (its coefficient × that year's momentum).

In [ ]:
yearly_bias = pd.concat({section: RESID[mask].groupby(races.loc[mask, "year"]).mean()
                         for section, mask in MASKS.items()}, axis=1)
pooled_bias = pd.DataFrame({section: RESID[mask].mean() for section, mask in MASKS.items()}).T.stack()
yearly_bias.loc["pooled"] = pooled_bias.reindex(yearly_bias.columns)
yearly_bias.loc["equal-year mean"] = yearly_bias.loc[EVAL_YEARS].mean()
yearly_bias.loc["mean |year bias|"] = yearly_bias.loc[EVAL_YEARS].abs().mean()
display(Markdown("### Mean residual (bias) by eval year and subset"))
display(yearly_bias.style.format(precision=2))

shift = PRED["A-NoMom"] - PRED["A"]
momentum_term = pd.Series({year: rolling["AB-15"][year].components["A_polling"].coefficients.loc[MOMENTUM, "coef"]
                           * year_level.loc[year, MOMENTUM] for year in EVAL_YEARS})
display(Markdown("### How dropping momentum moves each eval year's predictions (all valid races)"))
pd.DataFrame({"momentum": year_level.loc[EVAL_YEARS, MOMENTUM], "A's momentum term": momentum_term,
              "mean(A-NoMom − A)": shift.groupby(races["year"]).mean(),
              "sd within year": shift.groupby(races["year"]).std()}).style.format(precision=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, section in zip(axes, ["ALL", "EX_ANTE"]):
    for m in MODELS:
        ax.plot(EVAL_YEARS, yearly_bias.loc[EVAL_YEARS, (section, m)], STYLES[m], color=COLORS[m], lw=2, marker="o",
                ms=8, label=m)
    ax.axhline(0, color="#8d8c86", lw=1)
    ax.set_xticks(EVAL_YEARS)
    tidy(ax, f"National directional error by year ({section.lower()})", "eval year", "mean actual − predicted (pts)")
axes[0].legend(frameon=False)
plt.tight_layout()

## 6. Residual distributions

Pooled out-of-sample residuals for each subset. Skewness and excess kurtosis are 0 for a normal distribution, and `>10`, `>15`, and `>20` are the percentage of races missed by more than that many points. The *within-year sd* removes each year's national error first: what remains is the district-level spread, which momentum, being national, can't affect except through the other refit coefficients.

In [ ]:
def residual_stats(r, year):
    r = pd.Series(r).astype(float)
    q = r.quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    return {"n": len(r), "bias": r.mean(), "mae": r.abs().mean(), "rmse": np.sqrt((r ** 2).mean()), "sd": r.std(),
            "within-year sd": (r - r.groupby(year).transform("mean")).std(),
            "q05": q[0.05], "q25": q[0.25], "median": q[0.5], "q75": q[0.75], "q95": q[0.95],
            "skew": stats.skew(r), "excess_kurtosis": stats.kurtosis(r),
            **{f">{t}": 100 * (r.abs() > t).mean() for t in (10, 15, 20)}}


distribution = pd.DataFrame.from_dict({(section, m): residual_stats(RESID.loc[mask, m], races.loc[mask, "year"])
                                       for section, mask in MASKS.items() for m in MODELS}, orient="index")
distribution.index.names = ["subset", "model"]
distribution.style.format(precision=2).format({"n": "{:.0f}"})

## Conclusion

**Verdict:** dropping momentum doesn't make Model A more robust. Its only effect is to move each year's national level, and that level was more accurate *with* momentum in this backtest. Momentum's historical benefit is small but consistent in direction, while its coefficient is poorly determined. **Keep it in the established configurations**, and treat its magnitude as uncertain.

**1. Performance: dropping momentum is slightly worse in every subset.** Pooled MAE, current vs NoMom:

| subset | A | A-NoMom | AB-15 | AB-NoMom-15 |
|---|---|---|---|---|
| ex-ante competitive | 4.85 | 4.99 | 4.57 | 4.68 |
| ex-post competitive | 6.18 | 6.34 | 5.74 | 5.87 |
| all districts | 5.36 | 5.48 | 5.07 | 5.17 |

- **Winner calls:** A-NoMom nets −1 call against A (2 fixed, 3 broken). AB-NoMom-15 nets −3 against AB-15 (3 fixed, 6 broken), all in competitive races: −2 in 2018 and −1 in 2024.
- **Where the gain comes from:** almost entirely 2018 and 2020. In ex-ante races A-NoMom is worse by +0.35 (2018) and +0.18 (2020). In 2022 it is slightly better (−0.06), and in 2024 there is no difference (+0.01). NoMom never wins a year in the ensemble.

**2. The effect is purely national.**

- **Same district spread:** the within-year residual sd is unchanged (ex-ante 6.04 vs 6.04), and so are skewness and kurtosis.
- **A uniform shift:** dropping momentum moves every district in a year by almost the same amount (within-year sd of the shift ≤ 0.09 points). The shifts were −0.86 in 2018, +0.57 in 2020, −0.50 in 2022, and +0.05 in 2024.
- **Larger national error without it:** without momentum, the mean absolute national error rises from 1.36 to 1.86 points (all districts) and from 1.65 to 1.91 (ex-ante). The tails get slightly heavier too (misses over 10 points: 14.6% vs 13.8%).

**3. The negative coefficient is a collinearity split, identified from very few points.**

- **One value per election:** the generic ballot and momentum are national, and correlate at 0.92 across the training years of every window (VIF 6.3–7.1). After the intercept and the two slopes, only 3–6 year-level degrees of freedom remain.
- **The split:** without momentum, the generic-ballot coefficient is 0.79. With momentum, it rises to 1.31 and momentum takes −1.68. The two are trading off one national signal. Momentum's part is identified only from how far each year's late movement departs from what its ballot level predicts, which spans just ±1.5 points across nine elections. The year-level added-variable plot reproduces the −1.68 exactly. It is driven by 2008 (a late move toward R in a D year, and Democrats outperformed) against 2012.
- **Fragile to single elections:** dropping one training year moves the momentum coefficient over a 2.0–3.4-point range, and in the 2018 window dropping 2008 flips its sign (+0.79). The generic-ballot coefficient swings with it (A: range 0.7–1.2; A-NoMom: range 0.19–0.34).
- **Stable on the full window:** across the four expanding windows the coefficient stays between −1.33 and −1.68, and it is negative in every leave-one-year-out refit from the 2020 window on.
- **Overstated precision:** the row-level standard error (≈ 0.25) badly understates the true uncertainty.

**Recommendation.** On the stated priorities:

- **Competitive-race performance favors keeping momentum:** +0.11 to +0.16 MAE and −1 to −3 net calls without it.
- **Coefficient stability favors dropping it:** A-NoMom's generic-ballot coefficient is far steadier.
- **Parsimony favors dropping it.**

The performance edge is small and rests on about two elections, but it is consistent in direction and never materially reversed. Dropping momentum also didn't improve anything robustness is meant to protect: district-level dispersion, tails, and national bias are unchanged or worse. So the historical evidence *modestly* justifies keeping `gb_momentum_30d_raw` in `house_model_a` and `house_ensemble_ab_85_15`. Two cautions follow from the fragility:

- **Stage 2:** the national-error model should not rely on the momentum coefficient's row-level standard error. It is identified from nine elections, and one year can move it by two points.
- **2026:** report momentum's 2026 contribution (coefficient × 2026 momentum) separately in the forecast, so that an unusually large late move, which shifts every district by the same amount, is visible rather than buried.

All four eval years, 2024 included, were examined before, so this is development evidence, not a confirmatory test.